In [ ]:
import polars as pl

In [ ]:
from hexagonal.files.spec import get_polars_dataframe

In [ ]:
import gspread

In [ ]:
from hexagonal.cog import type_nom

In [ ]:
from pathlib import PurePath, Path

In [ ]:
type_nom.TYPES_NOMS

In [ ]:
types = [t.code for t in type_nom.TYPES_NOMS]
article = [t.article for t in type_nom.TYPES_NOMS]

In [ ]:
communes = get_polars_dataframe("data/02_clean/cog/communes.csv").select(
    code=pl.format("{type_commune}-{code_commune}"),
    nom_commune=pl.format("{}{}", pl.col("type_nom").replace_strict(types, article).str.to_uppercase(), pl.col("nom"))
)                   

In [ ]:
codes_remplace = {
    "COMD-29266": "COM-17415",
    "COM-11310": "COM-35238",
    "COM-11325": "COM-76540",
    "COM-62805": None,
}

In [ ]:
code_departement = pl.when(
    pl.col("Code INSEE commune").str.split("-").list.get(1).str.slice(0, 2).is_in(["97", "98"])
).then(
    pl.col("Code INSEE commune").str.split("-").list.get(1).str.slice(0, 3)
).otherwise(
    pl.col("Code INSEE commune").str.split("-").list.get(1).str.slice(0, 2)
).alias("Département")

In [ ]:
volontaires = pl.read_parquet("2026-07-01-volontaires.parquet").select(
    pl.col("created").alias("Date réponse"),
    pl.col('departement').alias('Département'),
    pl.col('email').alias('Email'),
    pl.col('last_name').alias('Nom'),
    pl.col('first_name').alias('Prénom'),
    pl.col('contact_phone').alias('Téléphone'),
    pl.col('commune-1781642735918-0').alias('Code INSEE commune').replace(list(codes_remplace.keys()), list(codes_remplace.values())),
    pl.col('nom_groupe').alias('Nom du groupe'),
    pl.when(
        pl.col('checkbox-group-1781533996600-0').list.contains("anim-ga")
    ).then(
        pl.lit("Animateur·rice")
    ).when(
        pl.col('checkbox-group-1781533996600-0').list.contains("membre-ga")
    ).then(
        pl.lit("Membre")
    ).alias('Rôle dans le GA'),
    pl.col('checkbox-group-1781643034799-0').list.get(0).alias('Expérience parrainage 2022'),
).join(communes, left_on="Code INSEE commune", right_on=["code"])

In [ ]:
export = volontaires.with_columns(
    code_departement, 
    pl.when(
        code_departement.is_in(["2A", "2B"])
    ).then(
        pl.lit("20")
    ).when(
        code_departement.is_in(["67", "68"])
    ).then(
        pl.lit("6AE")
    ).otherwise(
        code_departement
    ).alias("fichier")
).select(
    *volontaires.columns[:7], pl.col("nom_commune").alias("Nom de la commune"), *volontaires.columns[7:-1], "fichier"
)

In [ ]:
widths = {
    "Date réponse": 150,
    "Département": 100,
    "Email": 180,
    "Nom": 180,
    "Prénom": 150,
    "Téléphone": 150,
    "Code INSEE commune": 100,
    "Nom de la commune": 160,
    "Nom du groupe": 180,
    "Rôle dans le GA": 100,
    "Expérience parrainage 2022": 200,
}

In [ ]:
for f in export["fichier"].unique():
    ex = export.filter(pl.col("fichier") == f).select(pl.all().exclude("fichier"))
    ex.write_excel(
        f"out/volontaires/{f}.xlsx",
        column_widths=widths,
        header_format={
            "bold": True,
            "text_wrap": True,
        }
    )


In [ ]:
export

In [ ]:
from paramiko.client import SSHClient

client = SSHClient()
client.load_host_keys("/home/salome/Projets/playbooks/ssh_known_hosts")
client.connect("212.47.232.89", username="root")

sftp = client.open_sftp()

root = PurePath("/mnt/storage/data/__groupfolders/8/files")

for f in Path("out/volontaires").glob("*.xlsx"):
    dep = f.stem
    sftp.put(str(f), str(root / "Dossiers référents" / dep / f"volontaires.xlsx"))